# Newton's Method

## Learning Objectives
- Understand why Newton's method uses second-order information (curvature) to accelerate convergence over gradient ascent/descent
- Derive the Newton update $\theta \leftarrow \theta - H^{-1}\nabla J(\theta)$ from the second-order Taylor expansion of the objective
- Compute the Hessian $H = -X^T W X$ for logistic regression where $W_{ii} = h_i(1-h_i)$
- Understand **quadratic convergence** and why Newton's method needs far fewer iterations than gradient methods
- Implement `fit_newton` in NumPy for logistic regression using `np.linalg.solve`

## Problem Statement

Given a twice-differentiable objective $J(\theta)$ to maximise, find the root of its gradient $\nabla J(\theta^*) = 0$ using **Newton-Raphson** iteration.

Applied to logistic regression: given training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{n}$, maximise the log-likelihood $\ell(\theta)$ with update:

$\displaystyle \theta^{(t+1)} = \theta^{(t)} - H^{-1} \nabla \ell(\theta^{(t)})$

where $H = \nabla^2 \ell(\theta)$ is the **Hessian matrix**.

---

### Why Not Just Gradient Ascent?

Gradient ascent moves a fixed step $\alpha$ along the gradient, ignoring curvature:
- Slow convergence — needs many iterations
- Requires tuning the learning rate $\alpha$
- Steps are the same size regardless of the local landscape

Newton's method fits a **local quadratic** to $J$ at each step, then jumps directly to that quadratic's optimum:
- Automatic step size — scaled by the inverse Hessian
- **Quadratic convergence** near the optimum
- No learning rate hyperparameter

---

### Core Idea

At each iteration, model the objective as a paraboloid using the second-order Taylor expansion:

$\displaystyle J(\theta) \approx J(\theta_t) + \nabla J(\theta_t)^T (\theta - \theta_t) + \tfrac{1}{2}(\theta - \theta_t)^T H(\theta_t)(\theta - \theta_t)$

The minimum of this paraboloid is the next iterate.

| Property | Gradient Ascent | Newton's Method |
|---|---|---|
| Information used | Gradient $\nabla J$ only | Gradient $\nabla J$ + Hessian $H$ |
| Convergence rate | Linear | **Quadratic** |
| Learning rate | Required (tuned) | Not needed (implicit in $H^{-1}$) |
| Cost per iteration | $O(nd)$ | $O(nd^2 + d^3)$ |
| Iterations (logistic reg.) | $\sim$100–1000 | $\sim$5–15 |
| For linear regression | Many iterations | **Converges in 1 step** (Normal Equation) |

In [1]:
from IPython.display import SVG, display

svg = """
<svg xmlns="http://www.w3.org/2000/svg" width="620" height="310" viewBox="0 0 620 310">
  <rect width="620" height="310" fill="#fafafa" rx="8"/>
  <text x="310" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#222">Gradient Ascent vs Newton&#x2019;s Method</text>

  <!-- LEFT PANEL: Gradient Ascent - many small steps -->
  <text x="155" y="48" text-anchor="middle" font-size="11" font-weight="bold" fill="#555">Gradient Ascent &#x2014; Many Small Steps</text>
  <line x1="30" y1="260" x2="290" y2="260" stroke="#bbb" stroke-width="1"/>
  <line x1="30" y1="55"  x2="30"  y2="265" stroke="#bbb" stroke-width="1"/>

  <!-- Concave curve (log-likelihood shape) -->
  <path d="M 35,250 C 70,200 100,130 160,90 C 200,68 230,75 260,110 C 270,125 278,145 282,165"
        fill="none" stroke="#aaa" stroke-width="2"/>

  <!-- Many small gradient steps -->
  <circle cx="50"  cy="238" r="4" fill="#4a90d9"/>
  <circle cx="72"  cy="210" r="4" fill="#4a90d9"/>
  <circle cx="94"  cy="182" r="4" fill="#4a90d9"/>
  <circle cx="114" cy="155" r="4" fill="#4a90d9"/>
  <circle cx="132" cy="130" r="4" fill="#4a90d9"/>
  <circle cx="148" cy="108" r="4" fill="#4a90d9"/>
  <circle cx="160" cy="91"  r="4" fill="#4a90d9"/>
  <line x1="50"  y1="238" x2="72"  y2="210" stroke="#4a90d9" stroke-width="1.5"/>
  <line x1="72"  y1="210" x2="94"  y2="182" stroke="#4a90d9" stroke-width="1.5"/>
  <line x1="94"  y1="182" x2="114" y2="155" stroke="#4a90d9" stroke-width="1.5"/>
  <line x1="114" y1="155" x2="132" y2="130" stroke="#4a90d9" stroke-width="1.5"/>
  <line x1="132" y1="130" x2="148" y2="108" stroke="#4a90d9" stroke-width="1.5"/>
  <line x1="148" y1="108" x2="160" y2="91"  stroke="#4a90d9" stroke-width="1.5"/>

  <text x="45"  y="252" font-size="8" fill="#4a90d9">t=0</text>
  <text x="155" y="85"  font-size="8" fill="#4a90d9">t&#x2248;50&#x2026;</text>
  <text x="155" y="285" text-anchor="middle" font-size="10" fill="#cc3300">Slow &#x2014; many iterations, tuning &#x3B1; required</text>

  <!-- RIGHT PANEL: Newton's Method - few large jumps -->
  <text x="465" y="48" text-anchor="middle" font-size="11" font-weight="bold" fill="555">Newton&#x2019;s Method &#x2014; Quadratic Steps</text>
  <line x1="330" y1="260" x2="590" y2="260" stroke="#bbb" stroke-width="1"/>
  <line x1="330" y1="55"  x2="330" y2="265" stroke="#bbb" stroke-width="1"/>

  <!-- Same concave curve -->
  <path d="M 335,250 C 370,200 400,130 460,90 C 500,68 530,75 560,110 C 570,125 578,145 582,165"
        fill="none" stroke="#aaa" stroke-width="2"/>

  <!-- Local quadratic approximation at t=0 -->
  <path d="M 335,268 C 355,235 375,205 395,188 C 405,180 415,175 430,180"
        fill="none" stroke="#e07b00" stroke-width="1.5" stroke-dasharray="4,3"/>

  <!-- t=0 start -->
  <circle cx="345" cy="248" r="5" fill="#e05c5c"/>
  <text x="340" y="262" font-size="8" fill="#e05c5c">t=0</text>

  <!-- Jump to t=1 -->
  <circle cx="420" cy="178" r="5" fill="#e05c5c"/>
  <text x="424" y="176" font-size="8" fill="#e05c5c">t=1</text>
  <line x1="345" y1="248" x2="416" y2="182" stroke="#e05c5c" stroke-width="2"
        marker-end="url(#arr)"/>

  <!-- Jump to t=2 (near optimum) -->
  <circle cx="460" cy="90" r="5" fill="#1a6e2e"/>
  <text x="464" y="88" font-size="8" fill="#1a6e2e">t=2 &#x2248; &#x3B8;*</text>
  <line x1="420" y1="178" x2="456" y2="94" stroke="#e05c5c" stroke-width="2"/>

  <text x="455" y="78" font-size="9" fill="#1a6e2e">&#x2605;</text>
  <text x="465" y="285" text-anchor="middle" font-size="10" fill="#1a6e2e">Fast &#x2014; few iterations, no &#x3B1; needed</text>
</svg>
"""

display(SVG(svg))

## Update Rule

For a query point $\theta^{(t)}$, Newton's method computes the next iterate by jumping to the minimum of the local quadratic approximation:

$\displaystyle \theta^{(t+1)} = \theta^{(t)} - H(\theta^{(t)})^{-1}\, \nabla J(\theta^{(t)})$

where $H(\theta) = \nabla^2 J(\theta)$ is the **Hessian matrix** of second partial derivatives.

**Three forms for logistic regression** (maximising log-likelihood $\ell$):

| Form | Expression |
|------|------------|
| Scalar (1-param) | $\theta_{t+1} = \theta_t - \dfrac{\ell'(\theta_t)}{\ell''(\theta_t)}$ |
| Array (per-sample sum) | $\theta_{t+1} = \theta_t + \left[\sum_i x^{(i)}(x^{(i)})^T h_i(1-h_i)\right]^{-1} \sum_i (y^{(i)} - h_i) x^{(i)}$ |
| Matrix | $\theta_{t+1} = \theta_t + (X^T W X)^{-1} X^T(y - h)$ |

where $h = \sigma(X\theta) \in \mathbb{R}^n$ and $W = \mathrm{diag}(h_i(1-h_i)) \in \mathbb{R}^{n \times n}$.

**Key difference from gradient ascent**: The inverse Hessian $H^{-1}$ replaces (and subsumes) the learning rate $\alpha$, automatically scaling the step according to the local curvature.

## Derivation

**High-level steps:**
1. Apply Newton-Raphson to find the root of $\nabla \ell(\theta) = 0$
2. Expand $\nabla \ell$ around $\theta_t$ using a first-order Taylor expansion
3. Set the expansion to zero and solve for the next iterate
4. Specialise the gradient and Hessian to logistic regression

---

**Step 1 — Root-finding on the gradient**

At the optimum $\theta^*$, the gradient vanishes: $\nabla \ell(\theta^*) = 0$.  Newton-Raphson finds this root iteratively.

---

**Step 2 — First-order Taylor expansion of the gradient**

$\displaystyle \nabla \ell(\theta) \approx \nabla \ell(\theta_t) + H(\theta_t)(\theta - \theta_t)$

where $H(\theta_t) = \nabla^2 \ell(\theta_t)$ is the Hessian (the Jacobian of the gradient).

---

**Step 3 — Set to zero and solve**

$\displaystyle 0 = \nabla \ell(\theta_t) + H(\theta_t)(\theta_{t+1} - \theta_t)$

$\displaystyle \boxed{\theta_{t+1} = \theta_t - H(\theta_t)^{-1}\, \nabla \ell(\theta_t)}$

---

**Step 4 — Gradient and Hessian for logistic regression**

Recall (from the logistic regression derivation):

$\displaystyle \nabla \ell(\theta) = X^T(y - h), \quad h = \sigma(X\theta)$

Differentiating again with respect to $\theta$:

$\displaystyle H(\theta) = \frac{\partial}{\partial \theta} X^T(y - h) = -X^T \frac{\partial h}{\partial \theta} = -X^T W X$

where $W = \mathrm{diag}(h_i(1-h_i))$, using $\frac{\partial h_i}{\partial \theta} = h_i(1-h_i) x^{(i)}$ (sigmoid derivative).

Substituting into the Newton update:

$\displaystyle \theta_{t+1} = \theta_t - (-X^T W X)^{-1} X^T(y - h) = \theta_t + (X^T W X)^{-1} X^T(y - h)$

**Note**: $H$ is always **negative semi-definite** (since $W \succ 0$), confirming the log-likelihood is concave and Newton's method converges to the global maximum.

## Algorithm

Newton's method for logistic regression requires no learning rate and typically converges in 5–15 iterations.

**Step 1 — Initialise**

Set $\theta \leftarrow \mathbf{0} \in \mathbb{R}^{d+1}$. Prepend a bias column of ones to $X$.

**Step 2 — Compute predictions and weights**

$\displaystyle h = \sigma(X\theta) \in \mathbb{R}^n, \qquad w_i = h_i(1-h_i)$

**Step 3 — Compute gradient and Hessian**

$\displaystyle \nabla \ell = X^T(y - h) \in \mathbb{R}^{d+1}, \qquad H = -X^T W X \in \mathbb{R}^{(d+1)\times(d+1)}$

**Step 4 — Newton step** (solve the linear system instead of inverting)

$\displaystyle \theta \leftarrow \theta - H^{-1}\nabla\ell = \theta + (X^T W X)^{-1} X^T(y - h)$

Use `np.linalg.solve(X.T @ W @ X, X.T @ (y - h))` for numerical stability.

**Step 5 — Repeat** steps 2–4 until $\|\nabla\ell\| < \varepsilon$ or maximum iterations reached.

**Complexity**: $O(nd^2)$ to form $X^T W X$, plus $O(d^3)$ to solve the linear system per iteration. Total cost is dominated by $O(d^3)$ — feasible for $d < 10\,000$, impractical for neural networks.

| Quantity | Formula | Shape |
|----------|---------|-------|
| Predictions | $h = \sigma(X\theta)$ | $(n,)$ |
| Weights | $w_i = h_i(1-h_i)$ | $(n,)$ |
| Gradient | $g = X^T(y - h)$ | $(d+1,)$ |
| Hessian | $H = -X^T \mathrm{diag}(w) X$ | $(d+1, d+1)$ |
| Newton step | $\Delta\theta = -H^{-1}g$ | $(d+1,)$ |

## Key Properties

**Quadratic convergence** — near the optimum, the error halves in the exponent at each step: $\|\theta_{t+1} - \theta^*\| = O(\|\theta_t - \theta^*\|^2)$. This means if the error is $10^{-2}$, the next step gives $10^{-4}$, then $10^{-8}$.

**No learning rate** — the Hessian inverse automatically calibrates the step size. Too flat a region gets a large step; a highly curved region gets a small step.

**Linear regression special case** — the MSE Hessian $H = X^T X$ is constant everywhere (quadratic loss), so Newton's method converges in **exactly one step** from any starting point, recovering the Normal Equation: $\theta^* = (X^T X)^{-1} X^T y$.

**$O(d^3)$ cost** — Hessian inversion (or solving the linear system) costs $O(d^3)$ per iteration. Gradient ascent costs $O(nd)$ per iteration. Newton's method is only competitive when $d$ is small and/or very few iterations are required.

**Negative semi-definite Hessian** — for logistic regression, $H = -X^T W X \preceq 0$ everywhere (since $w_i > 0$). This guarantees the log-likelihood is globally concave and Newton's method always ascends.

**Relation to IRLS** — the Newton step $(X^T W X)^{-1} X^T(y-h)$ can be rewritten as a **weighted least squares** problem, which is the basis of the Iteratively Reweighted Least Squares (IRLS) algorithm used in GLM fitting.

**Quasi-Newton methods** — BFGS and L-BFGS approximate $H^{-1}$ using gradient differences across iterations, reducing cost to $O(d^2)$ or $O(md)$ per step while retaining superlinear convergence.

In [2]:
import numpy as np


def sigmoid(z):
    """
    Numerically stable sigmoid function.

    Input
    -----
    z : float or np.ndarray  — linear scores θᵀx (any shape)

    Output
    ------
    σ(z) : same shape as z, values in (0, 1)
    """
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-z)),
                    np.exp(z) / (1.0 + np.exp(z)))


def fit_newton(X, y, max_iter=20, tol=1e-8):
    """
    Train logistic regression via Newton's method (no learning rate required).

    Inputs
    ------
    X        : np.ndarray, shape (n, d+1)  — design matrix with bias column (x_0=1) prepended
    y        : np.ndarray, shape (n,)      — binary labels {0, 1}
    max_iter : int                         — maximum Newton iterations (typically 5–20 suffice)
    tol      : float                       — stop when ||∇ℓ|| < tol

    Outputs
    -------
    theta   : np.ndarray, shape (d+1,)   — learned parameters (bias first)
    history : list of (int, float)       — (iteration, log-likelihood) at each step
    """
    n, d = X.shape
    theta = np.zeros(d)
    history = []

    for t in range(max_iter):
        h   = sigmoid(X @ theta)                      # (n,)  predicted probabilities
        w   = h * (1 - h)                             # (n,)  diagonal weights W_ii = h_i(1-h_i)
        g   = X.T @ (y - h)                           # (d+1,) gradient  ∇ℓ = Xᵀ(y - h)
        H   = -(X.T * w) @ X                          # (d+1,d+1) Hessian H = -XᵀWX

        # Solve H Δθ = g  →  Δθ = -H⁻¹∇ℓ  (use solve for stability, not inv)
        delta = np.linalg.solve(-H, g)                # (d+1,)  Newton step
        theta = theta + delta                          # update

        h_clip = np.clip(h, 1e-12, 1 - 1e-12)
        ll = np.sum(y * np.log(h_clip) + (1 - y) * np.log(1 - h_clip))
        history.append((t, ll))

        if np.linalg.norm(g) < tol:
            break

    return theta, history


def predict_proba(X, theta):
    """
    Compute the probability of the positive class P(y=1 | x; theta).

    Inputs
    ------
    X     : np.ndarray, shape (n, d+1)  — design matrix with bias column (x_0=1) prepended
    theta : np.ndarray, shape (d+1,)    — model parameters

    Output
    ------
    proba : np.ndarray, shape (n,)  — P(y=1 | x; theta) for each example
    """
    return sigmoid(X @ theta)


def predict(X, theta, threshold=0.5):
    """
    Return binary class predictions.

    Inputs
    ------
    X         : np.ndarray, shape (n, d+1)  — design matrix with bias column (x_0=1) prepended
    theta     : np.ndarray, shape (d+1,)    — model parameters
    threshold : float                       — decision threshold (default 0.5)

    Output
    ------
    y_hat : np.ndarray, shape (n,)  — predicted labels {0, 1}
    """
    return (predict_proba(X, theta) >= threshold).astype(int)